In [ ]:
import pandas as pd
from sqlalchemy import create_engine

column_types = {
"osm_id": "str",
"addr_street": "str",
"addr_number": "str",
"grupa_parter": "str",
"funkcja_parter": "str",
"grupa_pietra": "str",
"funkcja_pietra": "str",
"liczba_kondygnacji": "Int64",
"mieszana": "boolean",
"rok_budowy_szac": "Int64",
"styl_arch": "str",
"stan_tech": "Int64",
"wysokosc_max": "float64",
"zabytek": "str"
}

In [ ]:
data_gdynia = pd.read_csv("../../data/raw/srodmiescie_raw.csv",
                          sep=";", index_col = 0, dtype = column_types)

data_clean = data_gdynia.dropna(thresh = 3)
data_gdynia.head()

In [ ]:
max_func_per_place = ((data_gdynia["funkcja_parter"].str.count(',')+1).max())
number_of_all_func = (data_clean["funkcja_parter"].str.count(',')+1).sum()
print(f'max_func_per_place: {max_func_per_place}')
print(f'number_of_all_func: {number_of_all_func}')

In [ ]:
data_extract = (
    data_clean
    .assign(funkcja_parter = lambda d: d['funkcja_parter'].str.replace('_',' ').str.lower().str.split(pat = ',') )
    .explode('funkcja_parter')
    .assign(funkcja_parter = lambda d: d['funkcja_parter'].str.strip())
)

In [ ]:
assert len(data_extract.index) == number_of_all_func

In [ ]:
data_comparison =(
    data_extract
    .groupby('funkcja_parter')
    .funkcja_parter.count()
    .reset_index(name = 'count')
    .rename(columns= {'funkcja_parter':'functions'})
)

In [ ]:
engine = create_engine("postgresql+psycopg://postgres@localhost:5432/Dataset_gdynia")

query = """
SELECT  
    TRIM(UNNEST(
        string_to_array(
            REPLACE(LOWER(dada.ground_floor_function),'_',' '),
            ','
        )
    )) AS functions, 
    COUNT(*) 
FROM dataset_gdynia_raw dada
GROUP BY functions;
"""

data_gdynia_sql = (
    pd.read_sql(query, engine)
    .sort_values('functions')
    .reset_index(drop = True)
)

In [ ]:
pd.testing.assert_frame_equal(data_gdynia_sql,data_comparison)

check if the sql export is the same as pandas